In [1]:
%load_ext rpy2.ipython

In [2]:
from pathlib import Path
from Bio import SeqIO
from rpy2 import robjects

In [3]:
%R pacman::p_load(ggtree, treeio, tidytree)

1,1,1


In [4]:
macse_input = Path('/home/panda2bat/Avivorous_bat/output/14_evolution-selective/input/aBSREL_input')
nt_lst = list(macse_input.rglob('*final_align_NT.aln'))

In [7]:
output = Path('/home/panda2bat/Avivorous_bat/output/14_evolution-selective/input/aBSREL_input/macse_handled')
output.mkdir(exist_ok=True, parents=True)

for nt in nt_lst:
    output_file = output / f"{nt.name.split('_')[0]}.fna"
    record_lst = list()
    for record in SeqIO.parse(nt, 'fasta'):
        record.id = record.id.split('|')[1]
        record.description = ''
        record_lst.append(record)
    SeqIO.write(record_lst, output_file, 'fasta')

for nt in nt_lst:
    output_file = output / f"{nt.name.split('_')[0]}.phy"
    records = {record.id:str(record.seq) for record in SeqIO.parse(nt, 'fasta')}
    first_record = records[list(records.keys())[0]]
    output_buff = open(output_file, 'w')
    output_buff.write(f"{len(list(records.keys()))} {len(records[list(records.keys())[0]])}\n")
    for key,value in records.items():
        output_buff.write(f"{key.split('|')[1]:14} {value}\n")
    output_buff.close()    

In [14]:
nt_lst

[PosixPath('/home/panda2bat/Avivorous_bat/output/14_evolution-selective/input/macse/OG0003945/OG0003945_final_align_NT.aln'),
 PosixPath('/home/panda2bat/Avivorous_bat/output/14_evolution-selective/input/macse/OG0005422/OG0005422_final_align_NT.aln'),
 PosixPath('/home/panda2bat/Avivorous_bat/output/14_evolution-selective/input/macse/OG0005276/OG0005276_final_align_NT.aln'),
 PosixPath('/home/panda2bat/Avivorous_bat/output/14_evolution-selective/input/macse/OG0005874/OG0005874_final_align_NT.aln'),
 PosixPath('/home/panda2bat/Avivorous_bat/output/14_evolution-selective/input/macse/OG0008582/OG0008582_final_align_NT.aln'),
 PosixPath('/home/panda2bat/Avivorous_bat/output/14_evolution-selective/input/macse/OG0006988/OG0006988_final_align_NT.aln'),
 PosixPath('/home/panda2bat/Avivorous_bat/output/14_evolution-selective/input/macse/OG0005581/OG0005581_final_align_NT.aln'),
 PosixPath('/home/panda2bat/Avivorous_bat/output/14_evolution-selective/input/macse/OG0005508/OG0005508_final_align_NT

In [24]:
raw_tree = '/home/panda2bat/Avivorous_bat/output/14_evolution-selective/input/aBSREL_input/rooted.tree'
all_species = [
	'ArtJam',
	'MyoMyo',
	'HipArm',
	'MinNat',
	'RouAeg',
	'MyoBra',
	'MyoLuc',
	'EptFus',
	'IaIo',
	'MolMol',
	'FelCat',
	'RhiFer',
	'HomSap',
	'PhyHas',
	'PhyDis',
	'DesRot',
	'PtePar',
	'NycAvi',
	'PipKuh',
	'EquCab',
	'StuHon',
	'MyoDav',
	'PteGig',
	'PteAle',
	'PteVam',
	'MusMus'
]
rconsole = robjects.r 
rconsole(f"""
         pacman::p_load(ggtree, treeio, tidytree)
         tree <- read.newick('{raw_tree}')
         """)

for nt in nt_lst:
    species_lst = list()
    output_file = output / f"{nt.name.split('_')[0]}.tree"
    for record in SeqIO.parse(nt, 'fasta'):
        species_lst.append(record.id.split('|')[1])
    drop_species = robjects.StrVector(list(set(species_lst)^set(all_species)))
    robjects.globalenv['drop_species'] = drop_species
    rconsole_cmd = f"""
    tree_reduce <- drop.tip(tree, drop_species)
    write.tree(tree_reduce, file = '{output_file}')
    """
    rconsole(rconsole_cmd)

    
    
    